In [18]:
from itertools import product
from dotenv import load_dotenv
import os

load_dotenv()

PUBTATOR_PATH = os.environ["PUBTATOR_PATH"]
MC_EMAIL = os.environ['MC_EMAIL']

import pandas as pd
import polars as pl
from tqdm.notebook import tqdm

# wags-llm imports
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner

# dgiLIT imports
from dgilit import (
    BioBertEntityTagger,
    CompositeEntityTagger,
    EntityPreTaggingService,
    InteractionClassificationPrompt,
    InteractionClassificationResult,
    PubTator3ChemicalTagger,
    SQLiteNormalizerCache,
    TaggerConfig,
    ViccNormalizer,
)

# Wags-LLM: Curated Set
Description

## PMID + Text Retrieval

Description

In [20]:
df = pd.read_csv('../../../data/2026-06-29-test2-RohitList.csv')
df

,pmid,drug_name,gene_name
0,22608338,DABRAFENIB,BRAF
1,22735384,DABRAFENIB,BRAF
2,38756640,DABRAFENIB,BRAF
3,24508103,VEMURAFENIB,BRAF
4,27460442,VEMURAFENIB,BRAF
...,...,...,...
102,25182707,SORAFENIB,KDR
103,35484400,SORAFENIB,KDR
104,3038326,ENALAPRIL MALEATE,ACE
105,31669155,LORLATINIB,ROS1


In [21]:
df = (
    df
    .drop_duplicates(subset=["pmid"])
    .reset_index(drop=True)
)


In [22]:
import requests
import xml.etree.ElementTree as ET
from time import sleep

def fetch_pubmed_abstracts(pmids, batch_size=100, email=None):
    """
    Fetch complete PubMed abstract text for PMIDs using NCBI E-utilities.
    Returns dict: {pmid: abstract_text}
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    abstracts = {}

    pmids = [str(pmid) for pmid in pmids if pd.notna(pmid)]

    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]

        params = {
            "db": "pubmed",
            "id": ",".join(batch),
            "retmode": "xml",
        }

        if email:
            params["email"] = email

        r = requests.get(base_url, params=params)
        r.raise_for_status()

        root = ET.fromstring(r.content)

        for article in root.findall(".//PubmedArticle"):
            pmid = article.findtext(".//PMID")

            abstract_parts = []
            for abstract_text in article.findall(".//Abstract/AbstractText"):
                label = abstract_text.attrib.get("Label")
                text = "".join(abstract_text.itertext()).strip()

                if text:
                    if label:
                        abstract_parts.append(f"{label}: {text}")
                    else:
                        abstract_parts.append(text)

            abstracts[str(pmid)] = "\n".join(abstract_parts) if abstract_parts else None

        sleep(0.34)  # stay under NCBI's 3 requests/second limit

    return abstracts


abstract_map = fetch_pubmed_abstracts(
    df["pmid"],
    email=MC_EMAIL
)

df["context"] = df["pmid"].astype(str).map(abstract_map)

df.to_excel('../../../data/2026-06-29-test2-RohitList-wText.xlsx')
df.head()

,pmid,drug_name,gene_name,context
0,22608338,DABRAFENIB,BRAF,BACKGROUND: Dabrafenib is an inhibitor of BRAF...
1,22735384,DABRAFENIB,BRAF,"BACKGROUND: Dabrafenib, an inhibitor of mutate..."
2,38756640,DABRAFENIB,BRAF,BACKGROUND: Gastrointestinal stromal tumor (GI...
3,24508103,VEMURAFENIB,BRAF,"BACKGROUND: In the BRIM-3 trial, vemurafenib w..."
4,27460442,VEMURAFENIB,BRAF,BACKGROUND: About half of patients with papill...


## Pre-tagging

In [6]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 1000

def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM interaction extraction task runner

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(InteractionClassificationPrompt(version="v2"))
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )

In [7]:
def classify_interaction(
    task_runner: StructuredTaskRunner,
    prompt: InteractionClassificationPrompt,
    context: str,
    candidate_drug: str,
    candidate_gene: str,
) -> InteractionClassificationResult:
    """Classify whether one candidate drug-gene pair is supported by biomedical text."""

    payload = prompt.build_payload(
        context=context,
        candidate_drug=candidate_drug,
        candidate_gene=candidate_gene,
    )

    try:
        task_result = task_runner.execute(
            prompt_name=prompt.name,
            prompt_version=prompt.version,
            payload=payload,
            response_model=InteractionClassificationResult,
        )

        return task_result

    except Exception as e:
        return InteractionClassificationResult(
            drug=candidate_drug,
            gene=candidate_gene,
            interaction=False,
            evidence=None,
            interaction_type=None,
            directionality=None,
            error_message=str(e),
        )

In [8]:
df = pl.read_excel("../../../data/2026-06-29-test2-RohitList-wText.xlsx") # file with pmids + context blocks

In [9]:
tagger = CompositeEntityTagger(
    BioBertEntityTagger(
        TaggerConfig(
            batch_size=16,
            include_drugs=True,
            include_genes=True,
            include_diseases=False,
            device=-1,
        )
    ),
    PubTator3ChemicalTagger.from_pubtator_file(PUBTATOR_PATH),
)

pretagger = EntityPreTaggingService(
    tagger=tagger,
    normalizer=ViccNormalizer(
        cache=SQLiteNormalizerCache(".dgilit_normalizer_cache.sqlite")
    ),
)

In [10]:
contexts = (
    df["context"]
    .fill_null("")
    .cast(pl.Utf8)
    .to_list()
)
pmids = df["pmid"].cast(str).to_list() if "pmid" in df.columns else [None] * len(contexts)
block_ids = [str(i) for i in range(len(contexts))]

tagged_blocks = pretagger.tag_blocks(
    contexts=contexts,
    pmids=pmids,
    block_ids=block_ids,
)

Device set to use cpu


Tagging text batches:   0%|          | 0/7 [00:00<?, ?it/s]

Device set to use cpu


Tagging text batches:   0%|          | 0/7 [00:00<?, ?it/s]

Normalizing unique entities:   0%|          | 0/542 [00:00<?, ?entity/s]

In [11]:
def entity_display_name(e):
    if e.concept and e.concept.concept_label:
        return e.concept.concept_label

    if e.text:
        return e.text

    return None


def build_candidate_context(block):
    drugs = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "drug"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    genes = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "gene"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    return {
        "pmid": block.pmid,
        "drugs": drugs,
        "genes": genes,
        "context": block.context,
    }

candidate = build_candidate_context(tagged_blocks[0])
candidate

{'pmid': '22608338',
 'drugs': ['dabrafenib'],
 'genes': ['BRAF'],
 'context': 'BACKGROUND: Dabrafenib is an inhibitor of BRAF kinase that is selective for mutant BRAF. We aimed to assess its safety and tolerability and to establish a recommended phase 2 dose in patients with incurable solid tumours, especially those with melanoma and untreated, asymptomatic brain metastases.\nMETHODS: We undertook a phase 1 trial between May 27, 2009, and March 20, 2012, at eight study centres in Australia and the USA. Eligible patients had incurable solid tumours, were 18 years or older, and had adequate organ function. BRAF mutations were mandatory for inclusion later in the study because of an absence of activity in patients with wild-type BRAF. We used an accelerated dose titration method, with the first dose cohort receiving 12 mg dabrafenib daily in a 21-day cycle. Once doses had been established, we expanded the cohorts to include up to 20 patients. On the basis of initial data, we chose a reco

## LLM Calls

Description

In [12]:
# EXPERIMENTAL
def normalized_entity_name(e):
    if e.concept and e.concept.concept_label and e.concept.concept_id:
        return e.concept.concept_label
    return None


def raw_entity_name(e):
    if e.text:
        return e.text
    return None


def get_candidates(block):
    """Normalized candidates only."""
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [normalized_entity_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [normalized_entity_name(e)]
        if name
    })

    return drugs, genes


def get_raw_candidates(block):
    """Raw mentions regardless of normalization."""
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [raw_entity_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [raw_entity_name(e)]
        if name
    })

    return drugs, genes

block = tagged_blocks[0]

normalized_drugs, normalized_genes = get_candidates(block)
raw_drugs, raw_genes = get_raw_candidates(block)

candidate_drugs = sorted(
    set(normalized_drugs) | set(raw_drugs)
)

candidate_genes = sorted(
    set(normalized_genes) | set(raw_genes)
)

candidate_drugs



# DEBUG
block = tagged_blocks[0]

normalized_drugs, _ = get_candidates(block)
raw_drugs, _ = get_raw_candidates(block)

print("Normalized:")
print(normalized_drugs)

print("\nRaw:")
print(raw_drugs)

print("\nRaw only:")
print(sorted(set(raw_drugs) - set(normalized_drugs)))

# DEBUG

search_term = "PENTOXIFYLLINE"

hits = []

for i, block in enumerate(tagged_blocks):
    normalized_drugs, _ = get_candidates(block)
    raw_drugs, _ = get_raw_candidates(block)

    all_drugs = set(normalized_drugs) | set(raw_drugs)

    if any(search_term.lower() in drug.lower() for drug in all_drugs):
        hits.append({
            "index": i,
            "pmid": getattr(block, "pmid", None),
            "normalized_drugs": normalized_drugs,
            "raw_drugs": raw_drugs,
        })

print(f"Found {len(hits)} blocks containing '{search_term}'")

for hit in hits:
    print("\n" + "=" * 80)
    print(f"PMID: {hit['pmid']}")
    print(f"Block index: {hit['index']}")
    print(f"Normalized: {hit['normalized_drugs']}")
    print(f"Raw: {hit['raw_drugs']}")

Normalized:
['dabrafenib']

Raw:
['Dabrafenib', 'dabrafenib']

Raw only:
['Dabrafenib']
Found 0 blocks containing 'PENTOXIFYLLINE'


In [13]:
import polars as pl
from itertools import product
from tqdm import tqdm


def make_pmid_to_genes(df: pl.DataFrame) -> dict:
    pmid_to_genes_df = (
        df
        .with_columns(pl.col("pmid").cast(pl.Utf8))
        .group_by("pmid")
        .agg(pl.col("gene_name").drop_nulls().unique().sort())
    )

    return dict(
        zip(
            pmid_to_genes_df["pmid"].to_list(),
            pmid_to_genes_df["gene_name"].to_list(),
        )
    )


def run_experiments(
    tagged_blocks,
    temperatures,
    num_runs,
    prompt_version: str,
    use_raw_drug_candidates: bool = False,
    seed_gene: str | None = None,
    pmid_to_genes: dict | None = None,
):
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            prompt = InteractionClassificationPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for block in tqdm(
                tagged_blocks,
                total=len(tagged_blocks),
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                normalized_drugs, normalized_genes = get_candidates(block)
                raw_drugs, raw_genes = get_raw_candidates(block)

                if use_raw_drug_candidates:
                    candidate_drugs = sorted(set(normalized_drugs) | set(raw_drugs))
                else:
                    candidate_drugs = normalized_drugs

                if pmid_to_genes is not None:
                    candidate_genes_for_classification = pmid_to_genes.get(
                        str(block.pmid),
                        [],
                    )
                elif seed_gene:
                    candidate_genes_for_classification = [seed_gene]
                else:
                    candidate_genes_for_classification = normalized_genes

                if not candidate_drugs or not candidate_genes_for_classification:
                    results.append(
                        {
                            "pmid": block.pmid,
                            "block_id": block.block_id,
                            "skipped": True,
                            "skip_reason": "No drug and gene candidates",
                            "candidate_drugs": candidate_drugs,
                            "candidate_genes": candidate_genes_for_classification,
                            "normalized_drugs": normalized_drugs,
                            "normalized_genes": normalized_genes,
                            "raw_drugs": raw_drugs,
                            "raw_genes": raw_genes,
                            "classifications": [],
                        }
                    )
                    continue

                classifications = []

                for candidate_drug, candidate_gene in product(
                    candidate_drugs,
                    candidate_genes_for_classification,
                ):
                    result = classify_interaction(
                        task_runner=task_runner,
                        prompt=prompt,
                        context=block.context,
                        candidate_drug=candidate_drug,
                        candidate_gene=candidate_gene,
                    )

                    classifications.append(result.model_dump())

                results.append(
                    {
                        "pmid": block.pmid,
                        "block_id": block.block_id,
                        "skipped": False,
                        "candidate_drugs": candidate_drugs,
                        "candidate_genes": candidate_genes_for_classification,
                        "normalized_drugs": normalized_drugs,
                        "normalized_genes": normalized_genes,
                        "raw_drugs": raw_drugs,
                        "raw_genes": raw_genes,
                        "num_candidate_pairs": len(candidate_drugs)
                        * len(candidate_genes_for_classification),
                        "classifications": classifications,
                    }
                )

            stored_runs.append(
                {
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "temperature": temp,
                    "use_raw_drug_candidates": use_raw_drug_candidates,
                    "seed_gene": seed_gene,
                    "used_pmid_to_genes": pmid_to_genes is not None,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

In [14]:
pmid_to_genes = make_pmid_to_genes(df)

stored_runs = run_experiments(
    tagged_blocks=tagged_blocks,
    temperatures=[0],
    num_runs=1,
    prompt_version="v2",
    use_raw_drug_candidates=False,
    seed_gene=None,
    pmid_to_genes=pmid_to_genes,
)

Running temp=0, run=1


Done temp=0, run=1


## Output

Description

In [17]:
seen = set()

for run in stored_runs:
    for result in run["results"]:
        key = (result["pmid"], tuple(result["candidate_genes"]))

        if key in seen:
            continue

        seen.add(key)

        print(
            f"PMID={result['pmid']} | "
            f"candidate_genes={result['candidate_genes']}"
        )

PMID=22608338 | candidate_genes=['BRAF']
PMID=22735384 | candidate_genes=['BRAF']
PMID=38756640 | candidate_genes=['BRAF']
PMID=24508103 | candidate_genes=['BRAF']
PMID=27460442 | candidate_genes=['BRAF']
PMID=37113357 | candidate_genes=['ALK']
PMID=28586279 | candidate_genes=['ALK']
PMID=38598794 | candidate_genes=['ALK']
PMID=23598171 | candidate_genes=['ALK']
PMID=25470694 | candidate_genes=['ALK']
PMID=25306392 | candidate_genes=['SMO']
PMID=25759019 | candidate_genes=['SMO']
PMID=28196207 | candidate_genes=['KIT']
PMID=28334365 | candidate_genes=['KIT']
PMID=21642685 | candidate_genes=['KIT']
PMID=21783417 | candidate_genes=['EGFR']
PMID=22285168 | candidate_genes=['EGFR']
PMID=22982650 | candidate_genes=['EGFR']
PMID=22452895 | candidate_genes=['EGFR']
PMID=25589191 | candidate_genes=['EGFR']
PMID=36108661 | candidate_genes=['RET']
PMID=31988000 | candidate_genes=['RET']
PMID=32846061 | candidate_genes=['RET']
PMID=29859851 | candidate_genes=['FLT3']
PMID=37116523 | candidate_gen

In [16]:
def save_experiment_results(
    stored_runs,
    csv_path="2026-06-29-interaction_classifications-rohit.csv",
    parquet_path="2026-06-29-interaction_classifications-rohit.parquet",
):
    rows = []

    for run in stored_runs:
        for result in run["results"]:
            candidate_drugs = result.get("candidate_drugs", [])
            candidate_genes = result.get("candidate_genes", [])
            classifications = result.get("classifications", [])

            normalized_drugs = result.get("normalized_drugs", [])
            normalized_genes = result.get("normalized_genes", [])
            raw_drugs = result.get("raw_drugs", [])
            raw_genes = result.get("raw_genes", [])

            used_pmid_to_genes = run.get("used_pmid_to_genes", False)

            base_row = {
                "run_idx": run.get("run_idx"),
                "prompt_version": run.get("prompt_version"),
                "temperature": run.get("temperature"),
                "used_raw_drug_candidates": run.get(
                    "use_raw_drug_candidates",
                    False,
                ),
                "seed_gene": run.get("seed_gene"),
                "used_pmid_to_genes": used_pmid_to_genes,
                "gene_source": "pmid_to_genes" if used_pmid_to_genes else (
                    "seed_gene" if run.get("seed_gene") else "normalized_genes"
                ),

                "pmid": result.get("pmid"),
                "block_id": result.get("block_id"),
                "skipped": result.get("skipped", False),
                "skip_reason": result.get("skip_reason"),

                # These are the actual candidates used for classification.
                "candidate_drugs": ";".join(candidate_drugs),
                "candidate_genes": ";".join(candidate_genes),
                "candidate_drug_count": len(candidate_drugs),
                "candidate_gene_count": len(candidate_genes),
                "num_candidate_pairs": result.get("num_candidate_pairs", 0),

                # These are tagger-derived candidates, kept for debugging.
                "normalized_drugs": ";".join(normalized_drugs),
                "normalized_genes": ";".join(normalized_genes),
                "normalized_drug_count": len(normalized_drugs),
                "normalized_gene_count": len(normalized_genes),

                "raw_drugs": ";".join(raw_drugs),
                "raw_genes": ";".join(raw_genes),
                "raw_drug_count": len(raw_drugs),
                "raw_gene_count": len(raw_genes),

                "result_error_message": result.get("error_message"),
            }

            if not classifications:
                rows.append(
                    {
                        **base_row,
                        "drug": None,
                        "gene": None,
                        "interaction": None,
                        "interaction_type": None,
                        "directionality": None,
                        "evidence": None,
                        "classification_error_message": None,
                        "error_message": result.get("error_message"),
                    }
                )
                continue

            for classification in classifications:
                rows.append(
                    {
                        **base_row,
                        "drug": classification.get("drug"),
                        "gene": classification.get("gene"),
                        "interaction": classification.get("interaction"),
                        "interaction_type": classification.get("interaction_type"),
                        "directionality": classification.get("directionality"),
                        "evidence": classification.get("evidence"),
                        "classification_error_message": classification.get(
                            "error_message"
                        ),
                        "error_message": classification.get("error_message")
                        or result.get("error_message"),
                    }
                )

    df_results = pl.DataFrame(rows)

    df_results.write_csv(csv_path)
    df_results.write_parquet(parquet_path)

    print(df_results.shape)
    return df_results


df_results = save_experiment_results(stored_runs)

df_results.head()

(206, 33)


run_idx,prompt_version,temperature,used_raw_drug_candidates,seed_gene,used_pmid_to_genes,gene_source,pmid,block_id,skipped,skip_reason,candidate_drugs,candidate_genes,candidate_drug_count,candidate_gene_count,num_candidate_pairs,normalized_drugs,normalized_genes,normalized_drug_count,normalized_gene_count,raw_drugs,raw_genes,raw_drug_count,raw_gene_count,result_error_message,drug,gene,interaction,interaction_type,directionality,evidence,classification_error_message,error_message
i64,str,i64,bool,null,bool,str,str,str,bool,str,str,str,i64,i64,i64,str,str,i64,i64,str,str,i64,i64,null,str,str,bool,str,str,str,str,str
0,"""v2""",0,false,null,true,"""pmid_to_genes""","""22608338""","""0""",false,null,"""dabrafenib""","""BRAF""",1,1,1,"""dabrafenib""","""BRAF""",1,1,"""Dabrafenib;dabrafenib""","""BRAF;BRAF kinase;mutant BRAF""",2,3,null,"""dabrafenib""","""BRAF""",true,"""inhibition""","""inhibitory""",null,null,null
0,"""v2""",0,false,null,true,"""pmid_to_genes""","""22735384""","""1""",false,null,"""dabrafenib;dacarbazine""","""BRAF""",2,1,2,"""dabrafenib;dacarbazine""","""BRAF""",2,1,"""Dabrafenib;dabrafenib;dacarbaz…","""BRAF;BRAF ( V600 );BRAF ( V600…",3,3,null,"""dabrafenib""","""BRAF""",true,"""inhibitor""","""inhibitory""",null,null,null
0,"""v2""",0,false,null,true,"""pmid_to_genes""","""22735384""","""1""",false,null,"""dabrafenib;dacarbazine""","""BRAF""",2,1,2,"""dabrafenib;dacarbazine""","""BRAF""",2,1,"""Dabrafenib;dabrafenib;dacarbaz…","""BRAF;BRAF ( V600 );BRAF ( V600…",3,3,null,"""dacarbazine""","""BRAF""",false,null,null,null,"""The abstract describes dacarba…","""The abstract describes dacarba…"
0,"""v2""",0,false,null,true,"""pmid_to_genes""","""38756640""","""2""",false,null,"""avapritinib;dabrafenib;imatini…","""BRAF""",6,1,6,"""avapritinib;dabrafenib;imatini…","""BRAF;KIT;PDGFRA""",6,3,"""avapritinib;dabrafenib;everoli…","""BRAF;BRAF V600E;BRAF V600E mut…",7,8,null,"""avapritinib""","""BRAF""",false,null,null,"""Avapritinib is mentioned only …",null,null
0,"""v2""",0,false,null,true,"""pmid_to_genes""","""38756640""","""2""",false,null,"""avapritinib;dabrafenib;imatini…","""BRAF""",6,1,6,"""avapritinib;dabrafenib;imatini…","""BRAF;KIT;PDGFRA""",6,3,"""avapritinib;dabrafenib;everoli…","""BRAF;BRAF V600E;BRAF V600E mut…",7,8,null,"""dabrafenib""","""BRAF""",true,"""inhibition""","""inhibitory""",null,null,null
